In [ ]:
%pip install earthscope-cli
%pip install boto3
%pip install earthscope-sdk 
%pip install obspy


In [ ]:
%pip install "setuptools<70.0.0"

In [ ]:
import os
# This tells the notebook EXACTLY where to find the 'es' command you just installed
os.environ['PATH'] = '/gpfs/fs1/home/cpenagon/.conda/envs/noisepy_meta/bin:' + os.environ['PATH']
!es login

In [ ]:
from earthscope_sdk import EarthScopeClient
from botocore.config import Config
import pandas as pd
import numpy as np 
import time
import os
import boto3
import gc 
from obspy.clients.fdsn import Client

# ==========================================
# 1. INITIALIZE CLIENTS
# ==========================================
print("Authenticating with EarthScope...")
client = EarthScopeClient()
creds = client.user.get_aws_credentials()

session = boto3.Session(
    aws_access_key_id=creds.aws_access_key_id,
    aws_secret_access_key=creds.aws_secret_access_key,
    aws_session_token=creds.aws_session_token,
)
s3_client = session.client("s3", config=Config(response_checksum_validation='when_required'))

BUCKET = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"
PREFIX = "miniseed/"


In [ ]:
client.user.get_profile()

In [ ]:
!curl ipinfo.io

In [ ]:
get_resp = s3_client.get_object(
    Bucket=BUCKET,
    #Key=f"{PREFIX}IU/2024/300/MBW.UW.2024.300#2",  # ~6 MB
    Key=f"{PREFIX}UW/2024/300/MBW.UW.2024.300#2", # ~6 MB
    #Key=f"{PREFIX}UW/2024/300/MPO.UW.2024.300#2",  # ~85 MB
   #Key=f"{PREFIX}UW/2024/300/SLA.UW.2024.300#2",  # ~400 MB
)

In [ ]:

# ==========================================
# 2. GET NETWORKS
# ==========================================
print("\nFetching network list from S3...")
list_resp = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, Delimiter="/")
nets = [c["Prefix"].split("/", 1)[1] for c in list_resp["CommonPrefixes"]]
network_list = [net.replace('/', '').strip() for net in nets if net.replace('/', '').strip()]

if 'SY' in network_list:
    network_list.remove('SY')
    print(" -> Dropped 'SY' (Synthetic Seismograms).")

# ==========================================
# 3. FETCH IRIS COORDINATES
# ==========================================
print(f"\nFetching EARTHSCOPE metadata for {len(network_list)} networks...")
client_iris = Client("https://service.earthscope.org")
station_data = []
batch_size = 50

for i in range(0, len(network_list), batch_size):
    batch = network_list[i : i + batch_size]
    net_string = ",".join(batch)

    try:
        inventory = client_iris.get_stations(network=net_string, level="station")
        for network in inventory:
            for station in network:
                end_time = station.end_date.datetime if station.end_date else pd.Timestamp('2100-01-01')
                station_data.append({
                    'network': network.code,
                    'station': station.code,
                    'lat': station.latitude,
                    'lon': station.longitude,
                    'start_date': station.start_date.datetime,
                    'end_date': end_time
                })
    except Exception:
        pass
    time.sleep(1)

df_coords = pd.DataFrame(station_data)
df_coords['start_date'] = pd.to_datetime(df_coords['start_date'])
df_coords['end_date'] = pd.to_datetime(df_coords['end_date'])
print(f" -> Success! Retrieved coordinates for {len(df_coords)} stations.")


In [ ]:
import os

# ==========================================
# 4. RAM-SAFE S3 GROUND-TRUTH SCAN 
# ==========================================
print("\nStarting RAM-Safe S3 Scan. This will take a while, but it will not crash...")

output_file = "s3_inventory.csv"

if os.path.exists(output_file):
    print(f"\n[!] Found existing '{output_file}'. Resuming progress...")
    try:
        existing_df = pd.read_csv(output_file, usecols=['network'])
        completed_networks = set(existing_df['network'].dropna().unique())
        
        network_list = [n for n in network_list if n not in completed_networks]
        print(f" -> Skipping {len(completed_networks)} completed networks.")
        print(f" -> {len(network_list)} networks remaining to process.")
    except Exception as e:
        print(f" -> Error reading existing CSV: {e}. Starting fresh might be required.")
else:
    print(f"\n[!] No existing inventory found. Starting fresh...")
    pd.DataFrame(columns=['network', 'station', 'year', 'yearday', 'lat', 'lon', 'dataacess_key']).to_csv(output_file, index=False)
# ------------------------

for network in network_list:
    print(f"\nScanning network: {network}...")
    datalist = []
    data_prefix = f"miniseed/{network}/"

    continuation_token = None
    is_truncated = True
    files_scanned = 0

    while is_truncated:
        try:
            kwargs = {'Bucket': BUCKET, 'Prefix': data_prefix}
            if continuation_token:
                kwargs['ContinuationToken'] = continuation_token

            response = s3_client.list_objects_v2(**kwargs)

            if "Contents" in response:
                for c in response["Contents"]:
                    key = c["Key"]
                    parts = key.split('/')
                    if len(parts) >= 5:
                        datalist.append([network, parts[-1].split('.')[0], parts[2], parts[3], key])
                        files_scanned += 1

            if files_scanned > 0 and files_scanned % 100000 == 0:
                print(f"  -> Processed {files_scanned:,} files...")

            is_truncated = response.get('IsTruncated', False)
            continuation_token = response.get('NextContinuationToken')

        except Exception as e:
            if "ExpiredToken" in str(e) or "Token expired" in str(e):
                print("  -> Token expired. Refreshing credentials silently...")
                creds = client.user.get_aws_credentials()
                session = boto3.Session(
                    aws_access_key_id=creds.aws_access_key_id,
                    aws_secret_access_key=creds.aws_secret_access_key,
                    aws_session_token=creds.aws_session_token,
                )
                s3_client = session.client("s3", config=Config(response_checksum_validation='when_required'))
            else:
                print(f"  -> Fatal error on {network}: {e}")
                break

    if datalist:
        df_chunk = pd.DataFrame(datalist, columns=['network', 'station', 'year', 'yearday', 'dataacess_key'])
        df_chunk['data_date'] = pd.to_datetime(df_chunk['year'] + df_chunk['yearday'].str.zfill(3), format='%Y%j')

        df_merged = pd.merge(df_chunk, df_coords, on=['network', 'station'], how='inner')
        df_exact = df_merged[
            (df_merged['data_date'] >= df_merged['start_date']) &
            (df_merged['data_date'] <= df_merged['end_date'])
        ].copy()

        df_exact = df_exact[['network', 'station', 'year', 'yearday', 'lat', 'lon', 'dataacess_key']]

        if not df_exact.empty:
            df_exact.to_csv(output_file, mode='a', header=False, index=False)
            print(f"  -> Saved {len(df_exact):,} verified records to disk.")
        else:
            print("  -> Skipped. Found files, but none matched valid IRIS epochs.")

        del df_chunk, df_merged, df_exact

    del datalist
    import gc
    gc.collect()

print(f"\n✅ All networks processed! Ground-truth inventory saved to {output_file}.")